# Bachelier's Theory of Speculation Revisited

## Interactive Reproduction Notebook

This notebook reproduces the main numerical experiments from the Alpha Stochastic Research repository:

**Bachelier (1900): Theory of Speculation — Reproducible Reconstruction of the Origins of Quantitative Finance**

The notebook is designed as an interactive companion to the paper and the tested Python source code.

It covers:

1. Bachelier arithmetic Brownian motion.
2. Martingale and variance-scaling checks.
3. Bachelier European call pricing.
4. Monte Carlo validation.
5. At-the-money square-root-of-time scaling.
6. Local comparison with Black-Scholes.

The notebook uses the tested source files in `src/` rather than duplicating the whole implementation.


## 1. Project Setup

This cell detects the repository root and makes the `src/` directory importable.

Run the notebook either from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Detect repository root.
current_dir = Path.cwd()

if (current_dir / "src").exists():
    ROOT_DIR = current_dir
elif (current_dir.parent / "src").exists():
    ROOT_DIR = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not find the repository root. "
        "Run this notebook from the repository root or from the notebooks/ directory."
    )

SRC_DIR = ROOT_DIR / "src"
FIGURES_DIR = ROOT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {ROOT_DIR}")
print(f"Source directory: {SRC_DIR}")
print(f"Figures directory: {FIGURES_DIR}")


## 2. Import the Tested Research Code

The notebook imports the same functions used by the scripts and tests.

This keeps the notebook aligned with the reproducible codebase.


In [ ]:
from brownian_motion import (
    P0,
    SIGMA,
    MATURITY,
    N_STEPS,
    N_PATHS,
    SEED,
    analyze_bachelier_paths,
    save_figure as save_brownian_figure,
    simulate_bachelier_paths,
)

from option_pricing import (
    K,
    N_PATHS as OPTION_N_PATHS,
    SEED as OPTION_SEED,
    bachelier_call_monte_carlo,
    bachelier_call_price,
    black_scholes_call_price,
    compare_with_black_scholes,
    compute_atm_scaling,
    save_figure as save_option_figure,
)

print("Research modules imported successfully.")


## 3. Bachelier Arithmetic Brownian Motion

The Bachelier model assumes that the price process follows:

$$
P_t = P_0 + \sigma W_t,
$$

where $W_t$ is a standard Brownian motion.

This implies:

$$
\mathbb{E}[P_t] = P_0,
$$

and

$$
\mathrm{Var}(P_t) = \sigma^2 t.
$$


In [ ]:
t_grid, paths = simulate_bachelier_paths(
    p0=P0,
    sigma=SIGMA,
    maturity=MATURITY,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    seed=SEED,
)

mean_path, empirical_variance, theoretical_variance = analyze_bachelier_paths(
    t_grid=t_grid,
    paths=paths,
    p0=P0,
    sigma=SIGMA,
)

summary = pd.DataFrame(
    {
        "Quantity": [
            "Initial price",
            "Arithmetic volatility",
            "Maturity",
            "Number of paths",
            "Number of time steps",
            "Terminal empirical mean",
            "Terminal empirical variance",
            "Theoretical terminal variance",
            "Maximum mean deviation from P0",
        ],
        "Value": [
            P0,
            SIGMA,
            MATURITY,
            N_PATHS,
            N_STEPS,
            mean_path[-1],
            empirical_variance[-1],
            theoretical_variance[-1],
            np.max(np.abs(mean_path - P0)),
        ],
    }
)

summary


## 4. Visual Check: Martingale and Variance Scaling

The left panel shows simulated Bachelier price paths.  
The right panel compares empirical variance with the theoretical variance $\sigma^2 t$.


In [ ]:
standard_deviation = SIGMA * np.sqrt(t_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for path in paths[:25]:
    axes[0].plot(t_grid, path, linewidth=0.8, alpha=0.35)

axes[0].plot(t_grid, mean_path, linewidth=2.5, label="Empirical mean")
axes[0].plot(
    t_grid,
    P0 + standard_deviation,
    linestyle="--",
    linewidth=1.5,
    label="P0 +/- one std. dev.",
)
axes[0].plot(t_grid, P0 - standard_deviation, linestyle="--", linewidth=1.5)

axes[0].set_title("Bachelier price paths")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Price")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_grid, empirical_variance, linewidth=2.0, label="Empirical variance")
axes[1].plot(
    t_grid,
    theoretical_variance,
    linestyle="--",
    linewidth=2.0,
    label="Theoretical variance",
)

axes[1].set_title("Variance scaling")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Bachelier arithmetic Brownian motion", fontsize=14)
fig.tight_layout()
plt.show()


## 5. Save the Brownian Motion Figure

This uses the repository function and saves the figure in `figures/`.


In [ ]:
brownian_figure_path = save_brownian_figure(
    t_grid=t_grid,
    paths=paths,
    mean_path=mean_path,
    empirical_variance=empirical_variance,
    theoretical_variance=theoretical_variance,
    output_path=FIGURES_DIR / "fig1_random_walk_martingale.png",
)

brownian_figure_path


## 6. Bachelier European Call Pricing

For a European call option with strike $K$ and maturity $T$, the payoff is:

$$
(P_T - K)^+.
$$

Under the Bachelier model:

$$
P_T = P_0 + \sigma \sqrt{T} Z,
\quad Z \sim \mathcal{N}(0,1).
$$

The Bachelier call price is:

$$
C = (P_0 - K)\Phi(d) + \sigma\sqrt{T}\phi(d),
$$

where:

$$
d = \frac{P_0-K}{\sigma\sqrt{T}}.
$$


In [ ]:
closed_form_price = bachelier_call_price(
    p0=P0,
    strike=K,
    sigma=SIGMA,
    maturity=MATURITY,
)

monte_carlo_price, monte_carlo_standard_error = bachelier_call_monte_carlo(
    p0=P0,
    strike=K,
    sigma=SIGMA,
    maturity=MATURITY,
    n_paths=OPTION_N_PATHS,
    seed=OPTION_SEED,
)

pricing_summary = pd.DataFrame(
    {
        "Quantity": [
            "Initial price",
            "Strike",
            "Arithmetic volatility",
            "Maturity",
            "Monte Carlo paths",
            "Closed-form price",
            "Monte Carlo price",
            "Monte Carlo standard error",
            "Absolute difference",
        ],
        "Value": [
            P0,
            K,
            SIGMA,
            MATURITY,
            OPTION_N_PATHS,
            closed_form_price,
            monte_carlo_price,
            monte_carlo_standard_error,
            abs(closed_form_price - monte_carlo_price),
        ],
    }
)

pricing_summary


## 7. At-the-Money Square-Root-of-Time Scaling

For an at-the-money call where $K=P_0$, the Bachelier price becomes:

$$
C_{ATM} = \sigma \sqrt{T}\phi(0).
$$

This shows that the at-the-money option value scales with $\sqrt{T}$.


In [ ]:
maturity_grid, atm_prices, theoretical_atm_prices = compute_atm_scaling(
    p0=P0,
    sigma=SIGMA,
)

atm_table = pd.DataFrame(
    {
        "Maturity": maturity_grid,
        "Bachelier ATM price": atm_prices,
        "Theoretical ATM price": theoretical_atm_prices,
    }
)

atm_table.head()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(maturity_grid, atm_prices, linewidth=2.2, label="Bachelier ATM price")
plt.plot(
    maturity_grid,
    theoretical_atm_prices,
    linestyle="--",
    linewidth=2.0,
    label="sigma sqrt(T) phi(0)",
)
plt.title("ATM option value scaling")
plt.xlabel("Maturity")
plt.ylabel("Call price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Local Comparison with Black-Scholes

The Black-Scholes model assumes proportional price changes:

$$
dS_t = \mu S_t dt + \sigma_{BS} S_t dW_t.
$$

The Bachelier model assumes absolute price changes.

For a local comparison near $S_0$, we match arithmetic volatility by:

$$
\sigma_{Bach} \approx \sigma_{BS} S_0.
$$


In [ ]:
strike_grid, bachelier_prices, black_scholes_prices = compare_with_black_scholes()

comparison_table = pd.DataFrame(
    {
        "Strike": strike_grid,
        "Bachelier price": bachelier_prices,
        "Black-Scholes price": black_scholes_prices,
        "Difference": bachelier_prices - black_scholes_prices,
    }
)

comparison_table.head()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(strike_grid, bachelier_prices, linewidth=2.2, label="Bachelier")
plt.plot(
    strike_grid,
    black_scholes_prices,
    linestyle="--",
    linewidth=2.0,
    label="Black-Scholes",
)
plt.title("Bachelier vs Black-Scholes")
plt.xlabel("Strike")
plt.ylabel("Call price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 9. Save the Option Pricing Figure

This saves the second figure used in the paper and README.


In [ ]:
option_figure_path = save_option_figure(
    maturity_grid=maturity_grid,
    atm_prices=atm_prices,
    theoretical_atm_prices=theoretical_atm_prices,
    strike_grid=strike_grid,
    bachelier_prices=bachelier_prices,
    black_scholes_prices=black_scholes_prices,
    output_path=FIGURES_DIR / "fig2_option_pricing.png",
)

option_figure_path


## 10. Reproducibility Checklist

This notebook confirms the main numerical content of the repository:

- The simulated Bachelier process has empirical mean close to $P_0$.
- Empirical variance follows $\sigma^2 t$.
- The Bachelier closed-form option price is validated by Monte Carlo simulation.
- At-the-money option prices scale with $\sqrt{T}$.
- Bachelier and Black-Scholes are locally close under a low-relative-volatility calibration.

For formal validation, run the test suite from the repository root:

```bash
pytest -q
```

For script-based reproduction:

```bash
python src/brownian_motion.py
python src/option_pricing.py
```
